In [33]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [34]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [35]:
query_com = """
SELECT DISTINCT c.Cliente,
       c.Nombres + ' ' + c.Apellidos AS Nombre,
       c.Email AS Email,
       c.Celular AS Celular,
       v.Fecha AS Fecha_Venta,
       v.Canal,
       SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2025-08-14'
    AND v.Fecha <= '2025-08-31'
    AND v.Venta_Neta > 0
GROUP BY  c.Cliente,
       c.Nombres + ' ' + c.Apellidos,
       c.Email,
       c.Celular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,C,EDWIN VERA,edwin.veralozano@gmail.com,3153857533,2025-08-20,Shopify,143691.3
1,C,EDWIN VERA,edwin.veralozano@gmail.com,3153857533,2025-08-23,Shopify,63022.5
2,C,EDWIN VERA,edwin.veralozano@gmail.com,3153857533,2025-08-25,Shopify,191588.4
3,C,EDWIN VERA,edwin.veralozano@gmail.com,3153857533,2025-08-26,Shopify,166379.4
4,C,EDWIN VERA,edwin.veralozano@gmail.com,3153857533,2025-08-29,Shopify,133607.7


In [36]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-08-09'
#     AND v.Fecha <= '2025-08-31'
#     AND Ubicacion = 'LOCCITANE SANTAFE MEDELLIN'
#     AND v.Venta_Neta >= 0
# GROUP BY  c.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [37]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta 
# 	,SUM(Venta_Neta) AS Venta, 
# 	Canal
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'CUMP_SEP2025')
# 	AND Fecha BETWEEN '2025-08-08' AND '2025-08-31'
# 	AND NOT Codigo_Descuento = '0002 Cumpleaños Empleados 50%'
# 	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
# GROUP BY  v.Cliente,
#        cc.Nombres + ' ' +  cc.Apellidos,
#        cc.Email,
#        cc.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [38]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clicks'
}, inplace=True)

# 1. Convertir la columna openedAt de string a lista real de fechas
df_campania['apertura'] = df_campania['apertura'].astype(str)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: ast.literal_eval(x) if x.startswith("[") else []
)

# 2. Expandir: cada fecha en una fila
df_expanded = df_campania.explode('apertura').copy()

# 3. Limpiar y convertir a datetime
df_expanded['apertura'] = df_expanded['apertura'].str.strip()
df_expanded['apertura'] = pd.to_datetime(
    df_expanded['apertura'],
    format='%Y-%m-%dT%H:%M:%S.%fZ',
    utc=True,
    errors='coerce'
).dt.date

#Convertir la columna clicks en Fecha
df_campania['clicks'] = df_campania['clicks'].astype(str)

def extraer_fecha_click(valor):
    try:
        # Convertir string a objeto Python (lista de dicts)
        lista = ast.literal_eval(valor)
        if isinstance(lista, list) and len(lista) > 0 and 'clicked_at' in lista[0]:
            fecha = pd.to_datetime(lista[0]['clicked_at'], utc=True, errors='coerce')
            return fecha.date()  # solo fecha
    except Exception:
        return pd.NaT
    return pd.NaT

df_campania['clicks'] = df_campania['clicks'].apply(extraer_fecha_click)

# 4. Normalizar correos
df_expanded['Email'] = df_expanded['Email'].str.lower().str.strip()

# 5. Elegir solo la primera apertura por email
df_abiertos = df_expanded.dropna(subset=['apertura']) \
    .sort_values('apertura') \
    .drop_duplicates(subset='Email', keep='first')

df_clics =  df_campania[df_campania['clicks'].notnull()].copy()

# Filtro correos abiertos hasta una fecha maxima
fecha_maxima = pd.to_datetime('2025-08-31').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'] <= fecha_maxima]
df_clicks_filtrados = df_clics[df_clics['clicks'] <= fecha_maxima]

total_abiertos = df_abiertos_filtrados['Email'].nunique()
total_clicks = df_clicks_filtrados['Email'].nunique()

TypeError: Invalid comparison between dtype=datetime64[ns] and date

In [ ]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_abiertos, on='Email', how='inner')

# Filtrar ventas dentro de 5 días desde la apertura
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['apertura']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")
print(f"Número de clientes que hicieron clics hasta {fecha_maxima}: {total_clicks}")
print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.0f}"  )
# Resultado final
df_atribucion.head()

Número de clientes que abrieron hasta 2025-08-31: 139
Número de clientes que hicieron clics hasta 2025-08-31: 2
Total de Ventas: 0
Total de venta atribuida: 0


,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,apertura,clicks,dias_post_apertura


In [ ]:
df_atribucion.to_excel('Ventas.xlsx', index=False)